In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# import seaborn as sns
from tda_methods import LSTMAutoencoder, GeometryConverter, PersistencePlotter, TakensEmbedding,\
      plot_3d_points, numpy_to_torch, window_2d_sequence
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
from scipy.stats import wasserstein_distance


In [ ]:
"Torus/Sphere data"

rng     = np.random.default_rng()
start_point, end_point, n_points = 0, 2*np.pi, 1000
u_angle = rng.uniform(start_point, end_point, n_points)
v_angle = rng.uniform(start_point, end_point, n_points)

# u, v = np.meshgrid(u, v)

R_major   = 1
r_tube    = R_major/4
converter = GeometryConverter(R_major, r_tube)
x, y, z   = converter.convert_angles_to_torus_xyz(u_angle, v_angle)
# # x, y, z   = converter.convert_angles_to_sphere_xyz(u_angle, v_angle)
noise = rng.normal(0, 0.04, n_points)
x += noise  # Add some noise to v
y += noise  # Add some noise to u
z += noise  # Add some noise to both
geo_coordinates = np.column_stack((x, y, z))  # shape (N, 3)

# x1,y1,z1,x2,y2,z2 = converter.convert_angles_to_2torus_xyz(u_angle, v_angle, d_shift=R_major)
# geo_coordinates   = np.vstack([np.column_stack((x1,y1,z1)), np.column_stack((x2,y2,z2))])
plot_3d_points((geo_coordinates[:, 0], geo_coordinates[:, 1], geo_coordinates[:, 2]))


In [ ]:
"persistence"

persistence    = PersistencePlotter(max_dim=1)
diagrams_list  = persistence.compute_persistence_diagrams(geo_coordinates)
diagrams_clean = persistence.remove_inf(diagrams_list)
entropy_array  = persistence.compute_entropy(diagrams_clean)
persistence_images_list = persistence.compute_persistence_image(diagrams_clean)
betti_curves_array      = persistence.compute_betti_curves(diagrams_clean)

print("Persistence array:", entropy_array)
print("Betti curves shape: (batch, homology_dim, filtration_steps) =", betti_curves_array.shape)

persistence.plot_persistence_diagrams(diagrams_list)
persistence.plot_entropy(entropy_array)
persistence.plot_persistence_image(persistence_images_list[1])  # H1
persistence.plot_betti_curves(betti_curves_array)


In [ ]:
"timedelay embeddings"

x  = np.linspace(0, 14*np.pi, 1000)
y1 = (0.5+0.5*x) * np.sin(2*x + 1) + 0.04 * rng.normal(size=x.shape)
y2 = 1.5*np.sin(2 * x) #+ 0.05 * (x-2)**1.2 #+ 0.02 * rng.normal(size=x.shape)
plt.figure()
plt.plot(x, y1, label='y1')
plt.plot(x, y2, label='y2')
plt.legend()
plt.show()

min_len = min(len(y1), len(y2))
y1, y2  = y1[:min_len], y2[:min_len]

time_delay, lag_dim = 20, 4
y1_delay_embeddings = make_timedelay_embeddings(y1, time_delay, lag_dim)
y1_now, y1_delayed  = y1_delay_embeddings[:, 0], y1_delay_embeddings[:, 1]

# time_delay2, lag_dim2 = 20, 4
y2_delay_embeddings = make_timedelay_embeddings(y2, time_delay, lag_dim)
y2_now, y2_delayed  = y2_delay_embeddings[:, 0], y2_delay_embeddings[:, 1]
plt.figure()
# plt.scatter(y1_now, y1_delayed, s=4)
plt.scatter(y2_now, y2_delayed, s=4)
plt.show()

# persistence stuff
persistence  = PersistencePlotter(max_dim=1)
n            = min(len(y1_delay_embeddings), len(y2_delay_embeddings))
y_all        = np.hstack((y1_delay_embeddings[:n], y2_delay_embeddings[:n]))
# y_data       = np.column_stack((y1_now, y1_delayed))
diagrams_list = persistence.compute_persistence_diagrams(y_all)
persistence.plot_persistence_diagrams(diagrams_list, title="y_all")


In [ ]:
"⚽️ Setting up data"
rng     = np.random.default_rng()

num_timesteps = 10_000
timesteps     = np.linspace(0, 20*np.pi, num_timesteps)
# X1 = np.sin(timesteps) * ((0.5+0.5*timesteps) * np.sin(2*timesteps + 1)) + 0.03 * rng.normal(size=timesteps.shape)
X1 = (2.0 * np.sin(0.2 * timesteps)) + 0.5 * np.sin(2*timesteps) + 0.02 * rng.normal(size=timesteps.shape)
X2 = np.cos(3*timesteps + 1) + 0.5 * np.sin(2*timesteps) + 0.02 * rng.normal(size=timesteps.shape)
X  = np.column_stack((X1, X2))

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

window_size     = 200
stride          = window_size//25
X_windows       = window_2d_sequence(X_scaled, window_size, stride)
X_train, X_test = train_test_split(X_windows, test_size=0.2, random_state=42, shuffle=False)

takens_object   = TakensEmbedding(delay=2, embedding_dim=3)
X_train_takens  = takens_object.make_timedelay_embeddings_grid(X_train)
X_test_takens   = takens_object.make_timedelay_embeddings_grid(X_test)

print("X shape", X.shape)
print("X_windows shape:", X_windows.shape)
print("X_train/test shape:", X_train.shape, X_test.shape)
# print("X_train_takens shape:", X_train_takens.shape)
# print("X_test_takens shape:", X_test_takens.shape)

plt.figure(figsize=(12, 5))
plt.plot(X1, label='Original Data')
plt.plot(X2, label='Original Data')


In [ ]:
"train LSTM encoder"

# X_train, X_test    = numpy_to_torch(X_train_takens), numpy_to_torch(X_test_takens)
X_train, X_test    = numpy_to_torch(X_train), numpy_to_torch(X_test)
input_dim          = X_train.shape[-1]
latent_dim         = 8
encoder_hidden_dim = 56
decoder_hidden_dim = 56
lstm_ae            = LSTMAutoencoder(input_dim, latent_dim, encoder_hidden_dim=encoder_hidden_dim,
                             decoder_hidden_dim=decoder_hidden_dim, epochs=100, learning_rate=0.001)
losses             = lstm_ae.train_model(X_train, patience=12)
device             = lstm_ae.device

# inference:
lstm_ae.eval()
with torch.no_grad():
    x_train_hat, z_train = lstm_ae(X_train.to(device))
    x_test_hat, z_test   = lstm_ae(X_test.to(device))

# visual test that encoder works on x_test
window_number, feature_number = 2, 2
plt.plot(x_test_hat[window_number, :, feature_number].cpu(), label=f'x{window_number}_hat')
plt.plot(X_test[window_number, :, feature_number].cpu(), label=f'x{window_number}')
plt.legend()
plt.show()


In [ ]:
"plotting latents"
from matplotlib.ticker import MaxNLocator

z_window_size   = z_train.shape[0]//3
z_stride        = z_window_size//6
z_train_windows = window_2d_sequence(z_train, z_window_size, z_stride)
print(z_train.shape, ", windowed becomes", z_train_windows.shape)

def plot_all_features_in_2d_array(dataset, window_number=0):
    """plots everything in 2d array. If array is 3d, plots features of a given input window"""
    plt.figure(figsize=(12, 7))

    # 3d array
    if dataset.ndim == 3 and dataset.shape[0] > 1:
        print(f"Plotting features for window {window_number} (dataset shape: {dataset.shape})")
        title = f"Features for window {window_number}"

        shape_idx_of_interest = 2 # 0 = windows, 1 = timesteps, 2 (or -1) = features
        for i in range(dataset.shape[shape_idx_of_interest]):
            plt.plot(dataset[window_number, :, i].cpu(), label=f'z{i}')

    # 2d array
    elif dataset.ndim == 2 or (dataset.ndim == 3 and dataset.shape[0] == 1):
        title = f"Features for entire array"

        shape_idx_of_interest = 1
        for i in range(dataset.shape[shape_idx_of_interest]):
            plt.plot(dataset[:, i].cpu(), label=f'z{i}')

    plt.title(title)
    plt.legend(fontsize=6)
    plt.show()

def plot_latent_evolution_grid(z: np.ndarray | torch.Tensor, max_latent_dims: int = 8, n_plots_per_row: int = 4):
    """Plot Heatmaps for latent tensors (W, C, L) as per-dimension heatmaps in a grid.
    Args:
        z: (W, C, L) array or tensor.
        max_latent_dims: max L to plot.
        n_plots_per_row: grid width.
    Note: Each subplot is independently scaled."""

    if isinstance(z, torch.Tensor):
        z = z.detach().cpu().numpy()

    L     = min(z.shape[2], max_latent_dims)
    nrows = int(np.ceil(L / n_plots_per_row))
    
    fig, axes = plt.subplots(
        nrows,
        n_plots_per_row,
        figsize=(4 * n_plots_per_row + 1, 3.6 * nrows),
        constrained_layout=True)
    axes = axes.flatten()

    im = None
    for i in range(L):
        im = axes[i].imshow(z[:, :, i], aspect='auto', cmap='hot')
        axes[i].set_title(f"z[{i}]")
        axes[i].yaxis.set_major_locator(MaxNLocator(integer=True))

        # remove per-plot axis labels
        axes[i].set_xlabel("")
        axes[i].set_ylabel("")

    for j in range(L, len(axes)):
        axes[j].axis('off')

    fig.subplots_adjust(right=0.88) # right-side colorbar

    cbar = fig.colorbar(
        im,
        ax=axes[:L],
        orientation='vertical',
        fraction=0.03,
        pad=0.02,
        shrink=0.9)
    cbar.set_label("activation")

    # global axis labels
    fig.text(0.5, -0.02, "chunk t", ha='center', va='bottom')
    fig.text(0.0, 0.5, "window #", va='center', rotation='vertical')

    # note
    fig.text(
        0.99,
        -0.02,
        "Colors are normalized per latent feature plot; not comparable across subplots.",
        ha='right',
        va='bottom',
        fontsize=8,
        alpha=0.7)

    fig.suptitle("z across windows + chunks", fontsize=16)
    plt.show()

plot_all_features_in_2d_array(z_train, window_number=1)
plot_latent_evolution_grid(z_train_windows, max_latent_dims=8, n_plots_per_row=4)

takens   = TakensEmbedding(delay=10, embedding_dim=3)
z_takens = takens.make_timedelay_embeddings_grid(z_train_windows)
plotter  = PersistencePlotter(max_dim=2, pixel_size=0.02)
dgms     = plotter.remove_inf(plotter.compute_persistence_diagrams(z_takens[0]))
plotter.plot_persistence_diagrams(dgms)


In [ ]:

window = 2
plot_3d_points((z_takens[window][:, 0], z_takens[window][:, 1], z_takens[window][:, 2]))
plot_3d_points((z_train_windows[window][:, 0], z_train_windows[window][:, 1], z_train_windows[window][:, 2]))


In [ ]:

z_betti    = plotter.compute_persistence_features(z_takens, mode="betti")
z_image    = plotter.compute_persistence_features(z_takens, mode="image")
z_diagram  = plotter.compute_persistence_features(z_takens, mode="diagram")
z_landscape= plotter.compute_persistence_features(z_takens, mode="landscape")

print("betti features shape:", z_betti.shape)
print("mean absolute difference:", np.mean(np.abs(z_betti[1:] - z_betti[:-1]))) # shape: (n_windows, H*W)

imgs = plotter.compute_persistence_images_global(plotter._cache_diagrams(z_takens))
print("std of persistence images:", np.std(imgs, axis=(1,2)))

plotter.plot_persistence_grid(z_takens)
plotter.plot_persistence_image_grid(z_takens)
plotter.plot_betti_grid(z_takens)
plotter.plot_landscape_grid(z_takens, num_steps=128, K_layers=3)


In [ ]:

def compute_wasserstein_distances(z_betti, z_image, z_diagram, z_landscape) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """computes Wasserstein distance between consecutive windows for each persistence feature type"""
    betti_dist_array     = np.array([wasserstein_distance(z_betti[i-1], z_betti[i]) for i in range(1, z_betti.shape[0])])
    image_dist_array     = np.array([wasserstein_distance(z_image[i-1], z_image[i]) for i in range(1, z_image.shape[0])])
    diagram_dist_array   = np.array([wasserstein_distance(z_diagram[i-1], z_diagram[i]) for i in range(1, z_diagram.shape[0])])
    landscape_dist_array = np.array([wasserstein_distance(z_landscape[i-1], z_landscape[i]) for i in range(1, z_landscape.shape[0])])
    return betti_dist_array, image_dist_array, diagram_dist_array, landscape_dist_array

betti_dist_array, image_dist_array, diagram_dist_array, landscape_dist_array = compute_wasserstein_distances(z_betti, z_image, z_diagram, z_landscape)
print("d_wasser(betti curves):", betti_dist_array)
print("d_wasser(persist. image):", image_dist_array)
print("d_wasser(persist. diagram):", diagram_dist_array)
print("d_wasser(persist. landscape):", landscape_dist_array)

# plt.plot(betti_dist_array, label='Betti Wasserstein Dist')
plt.plot(image_dist_array, label='Image Wasserstein Dist')
# plt.plot(diagram_dist_array, label='Diagram Wasserstein Dist')
# plt.legend()
# plt.show()
